# Carnet 1 : Benchmark Avancé des Modèles

## Contexte du Projet
Je cherche à prédire la **gravité des accidents de la route** (`grav_binary`) selon la météo, le type de route, les véhicules impliqués, etc. L'idée pour moi dans ce premier carnet est de lancer plusieurs modèles en même temps (un "benchmark") pour voir lequel s'en sort le mieux par défaut, avant de l'optimiser.

Comme on s'attaque à un problème de classification (prédire si un accident est grave ou non), je ne vais pas choisir un algorithme au hasard. L'idée d'un benchmark, c'est justement de tester plusieurs approches très différentes (de la simple régression logistique jusqu'aux modèles plus musclés de type « ensemble ») pour voir ce qui accroche le mieux à nos données. Ça me permet de justifier mes choix pour la suite plutôt que d'y aller à l'aveuglette !

**Critère de sélection : Le Recall**
Pour notre problématique de sécurité routière, ma priorité absolue est de maximiser la détection des accidents graves, même si cela implique une légère baisse de précision globale (on préfère une fausse alerte qu'un accident grave manqué). Nous porterons donc une attention particulière au **Recall**.

---

## Sommaire
* [Préparation](#Préparation)
* [Entraînement et Suivi (Tracking)](#Entraînement-et-Suivi-(Tracking))
* [Conclusion](#Conclusion)


## Préparation
### Imports et Configuration MLflow


In [1]:
import pandas as pd
import mlflow
import numpy as np
from datetime import datetime
from sklearn.metrics import log_loss
import logging
logging.getLogger("mlflow.sklearn").setLevel(logging.ERROR)
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# On connecte le notebook à notre base MLflow locale
TRACKING_URI = "http://localhost:5000"
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("InitialBenchmark")

2026/02/26 17:28:55 INFO mlflow.tracking.fluent: Experiment with name 'InitialBenchmark' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1772123335656, experiment_id='1', last_update_time=1772123335656, lifecycle_stage='active', name='InitialBenchmark', tags={}, workspace='default'>

### Chargement des données


In [2]:
df = pd.read_csv('../data/dataset_accident.csv', sep=';')
X = df.drop(columns=["grav_binary", "grav_ordered"])
y = df["grav_binary"]

print(f"Dimensions du dataset : {X.shape}")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Dimensions du dataset : (263356, 9)


## Entraînement et Suivi (Tracking)
### Lancement du Benchmark multi-modèles
Je vais entraîner 4 modèles classiques. MLflow va automatiquement se souvenir de leurs paramètres de base (ce qui m'évitera de les noter) et calculer 4 scores de performance pour que je puisse les comparer proprement.


In [3]:
models = {
    "LogisticRegressionBase": LogisticRegression(max_iter=1000),
    "RandomForestBase": RandomForestClassifier(n_estimators=50, random_state=42),
    "GradientBoostingBase": GradientBoostingClassifier(n_estimators=50, random_state=42),
    "XGBoostBase": xgb.XGBClassifier(n_estimators=50, max_depth=5, eval_metric='logloss', random_state=42)
}

print("-- Lancement du Benchmark --")

# Métadonnées dataset
classes = np.unique(y_train)
class_ratio = {str(c): round(np.sum(y_train == c) / len(y_train), 3) for c in classes}

for desc_name, model in models.items():
    with mlflow.start_run(run_name=f"{desc_name} {datetime.now().strftime('%d/%m/%Y %Hh:%Mm:%Ss')}") as run:
        print(f"\n-- Entraînement de {desc_name} --")
        
        # 0. On enregistre la configuration du modèle et les métadonnées du dataset
        mlflow.log_param("model_type", desc_name)
        mlflow.log_params(model.get_params())
        mlflow.log_param("dataset_train_size", len(X_train))
        mlflow.log_param("dataset_test_size", len(X_test))
        mlflow.log_param("n_features", X_train.shape[1])
        for cls, ratio in class_ratio.items():
            mlflow.log_param(f"class_ratio_{cls}", ratio)
        
        # 1. On entraîne
        model.fit(X_train, y_train)
        
        # 2. On fait les prédictions
        preds = model.predict(X_test)
        preds_proba = model.predict_proba(X_test)
        
        # 3. On calcule tous les scores
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, average='weighted')
        prec = precision_score(y_test, preds, average='weighted', zero_division=0)
        rec = recall_score(y_test, preds, average='weighted', zero_division=0)
        ll = log_loss(y_test, preds_proba)
        
        # 4. On envoie les notes à MLflow
        mlflow.log_metrics({
            "accuracy": acc,
            "f1_score": f1,
            "precision": prec,
            "recall": rec,
            "log_loss": ll
        })
        print(f"  Score recall : {rec:.4f} | f1 : {f1:.4f}")
        
        # 5. On place le modèle entraîné dans les artefacts de MLflow
        if isinstance(model, xgb.XGBClassifier):
            mlflow.xgboost.log_model(model, name=desc_name)
        else:
            mlflow.sklearn.log_model(model, name=desc_name)

print("\n-- Benchmark terminé. Résultats disponibles sur le serveur MLflow --")

-- Lancement du Benchmark --

-- Entraînement de LogisticRegressionBase --
  Score recall : 0.6679 | f1 : 0.6428
🏃 View run LogisticRegressionBase 26/02/2026 17h:28m:55s at: http://localhost:5000/#/experiments/1/runs/f4ad288ded484d85b3e95dd1e950940d
🧪 View experiment at: http://localhost:5000/#/experiments/1

-- Entraînement de RandomForestBase --
  Score recall : 0.7186 | f1 : 0.6933
🏃 View run RandomForestBase 26/02/2026 17h:29m:07s at: http://localhost:5000/#/experiments/1/runs/9b9c83d5f1de4d66911d9aabe28f4b27
🧪 View experiment at: http://localhost:5000/#/experiments/1

-- Entraînement de GradientBoostingBase --
  Score recall : 0.7188 | f1 : 0.6849
🏃 View run GradientBoostingBase 26/02/2026 17h:29m:24s at: http://localhost:5000/#/experiments/1/runs/58e46ee3218e446598046fcfae9682b6
🧪 View experiment at: http://localhost:5000/#/experiments/1

-- Entraînement de XGBoostBase --
  Score recall : 0.7203 | f1 : 0.6931
🏃 View run XGBoostBase 26/02/2026 17h:29m:43s at: http://localhost:5000

## Conclusion
### Choix du meilleur modèle pour la suite

En analysant [les graphiques des métriques sur MLflow](http://127.0.0.1:5000/#/experiments/9/models?viewMode=CHART), je vois bien que **XGBoost** est majoritairement le meilleur, en particulier sur ma métrique clé : le **Recall**. Même s'il peut être légèrement moins bien que GradientBoosting sur la précision, sa capacité à identifier efficacement les accidents graves en fait le modèle le plus solide pour mon cas d'usage.

Je pars donc sur XGBoost comme champion pour notre prochain notebook de tuning !

---
### Prochaine étape :
Génération d'artefacts visuels et recherches d'hyperparamètres sur ce XGBoost.
[-- Ouvrir le Carnet 2 : Tuning Manuel et Artefacts --](02_Tuning_Manuel_Artefacts.ipynb)
